# Домашнє завдання: Внесення оновлень в БД і робота з транзакціями

Це ДЗ передбачене під виконання на локальній машині. Виконання з Google Colab буде суттєво ускладнене.

## Підготовка
1. Переконайтесь, що у вас встановлені необхідні бібліотеки:
   ```bash
   pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv
   ```

2. Створіть файл `.env` з параметрами підключення до бази даних classicmodels. Базу даних ви можете отримати через

  - docker-контейнер згідно існтрукції в [документі](https://www.notion.so/hannapylieva/Docker-1eb94835849480c9b2e7f5dc22ee4df9), також відео інструкції присутні на платформі - уроки "MySQL бази, клієнт для роботи з БД, Docker і ChatGPT для запитів" та "Як встановити Docker для роботи з базами даних без терміналу"
  - або встановивши локально цю БД - для цього перегляньте урок "Опціонально. Встановлення MySQL та  БД Сlassicmodels локально".
  
  Приклад `.env` файлу ми створювали в лекції. Ось його обовʼязкове наповнення:
    ```
    DB_HOST=your_host
    DB_PORT=3306 або 3307 - той, який Ви налаштували
    DB_USER=your_username
    DB_PASSWORD=your_password
    DB_NAME=classicmodels
    ```
  Якщо ви створили цей файл під час перегляду лекції - **новий створювати не треба**. Замініть лише назву БД, або пропишіть назву в коді створення підключення (замість отримання назви цільової БД зі змінних оточення). Але переконайтесь, що до `.env` файл лежить в тій самій папці, що і цей ноутбук.

  **УВАГА!** НЕ копіюйте скрит для **створення** `.env` файлу. В лекції він наводиться для прикладу. І давалось пояснення, що в реальних проєктах ми НІКОЛИ не пишемо доступи до бази в коді. Копіювання скрипта для створення `.env` файлу сюди в ДЗ буде вважатись грубою помилкою і ми зніматимемо бали.

3. Налаштуйте підключення через SQLAlchemy до БД за прикладом в лекції.

Рекомендую вивести (відобразити) змінну engine після створення. Вона має бути не None! Якщо None - значить у Вас не підтягнулись налаштування з .env файла.

Ви також можете налаштувати параметри підключення до БД без .env файла, просто прописавши текстом в відповідних місцях. Це - не рекомендований підхід.


In [1]:
import datetime
import os
import sqlalchemy as sa
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import logging

In [2]:
def create_connection():
    """
    Створює підключення через SQLAlchemy
    """
    # Завантажуємо змінні середовища
    load_dotenv()

    # Отримуємо параметри з environment variables
    host = os.getenv('DB_HOST', '127.0.0.1')
    port = os.getenv('DB_PORT', '3306')
    user = os.getenv('DB_USER')
    password = os.getenv('DB_PASSWORD')
    database = os.getenv('DB_NAME')

    if not all([user, password, database]):
        raise ValueError("Не всі параметри БД задані в .env файлі!")

    # Створюємо connection string
    connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"

    # Створюємо engine з connection pooling
    engine = create_engine(
        connection_string,
        pool_size=2,            # Розмір пулу підключень
        max_overflow=20,        # Максимальна кількість додаткових підключень
        pool_pre_ping=True,     # Перевірка підключення перед використанням
        echo=False              # Логування SQL запитів (True для debug)
    )

    # Тестуємо підключення
    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT 1"))
            result.fetchone()

        print("✅ Підключення до БД успішне!")
        print(f"🔗 host={host}, port={port}, db={database}")
        
        return engine

    except Exception as e:
        print(f"❌ Помилка підключення: {e}")
        return None

# Створюємо підключення
engine = create_connection()

✅ Підключення до БД успішне!
🔗 host=127.0.0.1, port=3306, db=classicmodels



## Завдання

### Завдання 1: Оновлення інформації про клієнта (2 бали)

**Створіть функцію для оновлення контактної інформації клієнта за його номером** з наступними можливостями:
- Оновлення телефону клієнта
- Оновлення email (якщо поле існує в таблиці)

Опціонально, якщо вам хочеться більше практики:
- Логування змін в окрему таблицю

Використайте підхід з параметризованими запитами через `text()` та `UPDATE` оператор. Не забудьте на початку перевірити чи існує клієнт з таким номером в базі - це хороша практика.

Отримати всі колонки, які існують в таблиці ви можете наступним запитом
```
  SELECT COLUMN_NAME, DATA_TYPE
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_NAME = 'customers'
```

Запустіть функцію і продемонструйте її роботу, запустивши SELECT, який допоможе це зробити.

In [3]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

def update_customers_contact_info(customerNumber, new_phone):
    """
    Оновлення телефону клієнта в таблиці customers з використанням транзакції
    """

    # Перевірка чи існує клієнт (окремо від транзакції)
    check_query = text("""
        SELECT customerNumber, customerName, phone
        FROM customers
        WHERE customerNumber = :customerNumber
    """)
    
    update_query = text("""
        UPDATE customers
        SET phone = :new_phone
        WHERE customerNumber = :customerNumber
    """)
    
    try:
        with engine.connect() as conn:
            customer = conn.execute(
                check_query,
                {"customerNumber": customerNumber}
            ).fetchone()
    
            if not customer:
                logger.error(f"❌ Клієнт {customerNumber} не знайдений")
                return False
    
            logger.info(f"👤 Клієнт: {customer.customerName}")
            logger.info(f"📞 Старий телефон: {customer.phone}")

        with engine.begin() as conn:
            conn.execute(update_query, {
                    "customerNumber": customerNumber,
                    "new_phone": new_phone
                })
        logger.info(f"✅ Новий телефон: {new_phone}")
        logger.info("Оновлення виконано успішно")
        return True

    except Exception as e:
        logger.error(f"❌ Помилка при зміні: {e}")
        return False

# Тестуємо оновлення інформації
customerNumber = 103
success = update_customers_contact_info(
    customerNumber,
    new_phone='40.32.2557'
)

INFO: 👤 Клієнт: Atelier graphique
INFO: 📞 Старий телефон: 40.32.2556
INFO: ✅ Новий телефон: 40.32.2557
INFO: Оновлення виконано успішно


In [4]:
# Після виконання UPDATE перевірка результат запитом SELECT

with engine.connect() as conn:
    result = conn.execute(
        text("""
            SELECT customerNumber, customerName, phone
            FROM customers
            WHERE customerNumber = :customerNumber
        """),
        {"customerNumber": customerNumber}
    )

    updated_customer = result.fetchone()

print("Результат після оновлення:")
print(f"ID клієнта: {updated_customer.customerNumber}")
print(f"Назва клієнта: {updated_customer.customerName}")
print(f"Телефон: {updated_customer.phone}")

Результат після оновлення:
ID клієнта: 103
Назва клієнта: Atelier graphique
Телефон: 40.32.2557


### Завдання 2: Створення нового замовлення з транзакцією (5 балів)

**Реалізуйте процес створення нового замовлення** з наступними кроками в одній транзакції:
- Створення запису в таблиці `orders`
- Додавання товарних позицій в `orderdetails`
- Перевірка наявності товарів на складі
- Зменшення кількості товарів на складі

Запустіть процес з тестовими даними і продемонструйте через SELECT, що процес успішно відпрацював і були виконані необхідні операції.




In [5]:
def add_new_order(orderNumber, orderDate, requiredDate, shippedDate, status, comments,
                  customerNumber, productCode, quantityOrdered, priceEach, orderLineNumber):
    """
    Cтворення нового замовлення з використанням транзакції
    """

    check_customer_query = text("""
        SELECT customerNumber, customerName
        FROM customers
        WHERE customerNumber = :customerNumber
    """)

    check_product_query = text("""
        SELECT productCode, productName, quantityInStock
        FROM products
        WHERE productCode = :productCode
    """)
    
    insert_order_query = text("""
        INSERT INTO orders (orderNumber, orderDate, requiredDate, shippedDate, status, comments, customerNumber)
        VALUES (:orderNumber, :orderDate, :requiredDate, :shippedDate, :status, :comments, :customerNumber)
    """)

    insert_orderdetails_query = text("""
        INSERT INTO orderdetails (orderNumber, productCode, quantityOrdered, priceEach, orderLineNumber)
        VALUES (:orderNumber, :productCode, :quantityOrdered, :priceEach, :orderLineNumber)
    """)

    update_stock_query = text("""
        UPDATE products
        SET quantityInStock = quantityInStock - :quantityOrdered
        WHERE productCode = :productCode
    """)

    try:
        with engine.connect() as conn:
            customer = conn.execute(check_customer_query,
                {"customerNumber": customerNumber}).fetchone()

            if not customer:
                logger.error(f"❌ Клієнт {customerNumber} не знайдений")
                return False

            product = conn.execute(check_product_query, {"productCode": productCode}).fetchone()

            if not product:
                logger.error(f"❌ Товар {productCode} не знайдений")
                return False

            if product.quantityInStock < quantityOrdered:
                logger.error(f"❌ Недостатньо товару на складі. Є: {product.quantityInStock}, потрібно: {quantityOrdered}")
                return False

            logger.info(f"Клієнт: {customer.customerName}")
            logger.info(f"Товар: {product.productName}")
            logger.info(f"Залишок до оновлення: {product.quantityInStock}")

        with engine.begin() as conn:
            conn.execute(
                insert_order_query,
                {
                    "orderNumber": orderNumber,
                    "orderDate": orderDate,
                    "requiredDate": requiredDate,
                    "shippedDate": shippedDate,
                    "status": status,
                    "comments": comments,
                    "customerNumber": customerNumber
                }
            )
            logger.info("✅ Крок 1: Додано запис у orders")

            conn.execute(
                insert_orderdetails_query,
                {
                    "orderNumber": orderNumber,
                    "productCode": productCode,
                    "quantityOrdered": quantityOrdered,
                    "priceEach": priceEach,
                    "orderLineNumber": orderLineNumber
                }
            )
            logger.info("✅ Крок 2: Додано запис у orderdetails")

            conn.execute(
                update_stock_query,
                {
                    "productCode": productCode,
                    "quantityOrdered": quantityOrdered
                }
            )
            logger.info("✅ Крок 3: Оновлено залишок на складі")

        logger.info("✅Нове замовлення створено успішно")
        return True

    except Exception as e:
        logger.error(f"❌ Помилка при створенні замовлення: {e}")
        return False
 
# Тестуємо створення замовлення
success = add_new_order(
    orderNumber=10428,
    orderDate='2005-06-01',
    requiredDate='2005-06-10',
    shippedDate='2005-06-03',
    status='Shipped',
    comments='Test order',
    customerNumber=103,
    productCode='S10_1678',
    quantityOrdered=5,
    priceEach=95.70,
    orderLineNumber=1
)

INFO: Клієнт: Atelier graphique
INFO: Товар: 1969 Harley Davidson Ultimate Chopper
INFO: Залишок до оновлення: 7923
INFO: ✅ Крок 1: Додано запис у orders
INFO: ✅ Крок 2: Додано запис у orderdetails
INFO: ✅ Крок 3: Оновлено залишок на складі
INFO: ✅Нове замовлення створено успішно


In [6]:
# Перевірка таблиці orders
with engine.connect() as conn:
    result = conn.execute(
        text("""
            SELECT orderNumber, orderDate, requiredDate, shippedDate, status, comments, customerNumber
            FROM orders
            WHERE orderNumber = :orderNumber
        """),
        {"orderNumber": 10426}
    )
    
print(result.fetchone())

(10426, datetime.date(2005, 6, 1), datetime.date(2005, 6, 10), datetime.date(2005, 6, 3), 'Shipped', 'Test order', 103)


In [7]:
# Перевірка таблиці orderdetails

with engine.connect() as conn:
    result = conn.execute(
        text("""
            SELECT orderNumber, productCode, quantityOrdered, priceEach, orderLineNumber
            FROM orderdetails
            WHERE orderNumber = :orderNumber
        """),
        {"orderNumber": 10426}
    )
    
print(result.fetchone())

(10426, 'S10_1678', 5, Decimal('95.70'), 1)


In [8]:
# Перевірка залишку в products

with engine.connect() as conn:
    result = conn.execute(
        text("""
            SELECT productCode, productName, quantityInStock
            FROM products
            WHERE productCode = :productCode
        """),
        {"productCode": "S10_1678"}
    )
    
print(result.fetchone())

('S10_1678', '1969 Harley Davidson Ultimate Chopper', 7918)


**Time spent — 6 годин**

**What I learned:**

- Вносити оновлення в БД через SQLAlchemy.
- Додавати нові записи та змінювати існуючі.
- Працювати з транзакціями для безпечного виконання змін.
- Перевіряти коректність даних перед оновленням.
- Узгоджено змінювати дані в кількох пов’язаних таблицях.
- Контролювати результат змін через SELECT.
- Логувати успішні операції та помилки.